In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# 1. NAČTENÍ DAT
# ==========================================
# Cesta k datům (z lokální složky, která se nepushuje na Git!)
df = pd.read_csv('../data/student_dropout_dataset_v3.csv')

# Definice X (prediktory) a y (cíl)
# Zde si případně upravíte podmnožinu dat nebo odvodíte nový cíl podle zadání
X = df.drop('Dropout', axis=1)
y = df['Dropout']

# ==========================================
# 2. ROZDĚLENÍ DAT (Kritický krok před úpravami)
# ==========================================
# Tímto splníme zadání: trénovací množina je větší než testovací
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 3. DEFINICE PŘEDZPRACOVÁNÍ (ColumnTransformer)
# ==========================================
# ZDE DOPLNÍ DATOVÝ INŽENÝR (Matěj/Kozub) NÁZVY SLOUPCŮ:
numeric_features = ['Age', 'GPA', 'Study_Hours_per_Day'] # (příklad, doplňte zbytek)
categorical_features = ['Gender', 'Department'] # (příklad, doplňte zbytek)

# Jak se zpracují čísla (doplnění chybějících + škálování)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Jak se zpracují kategorie (doplnění chybějících + převod na čísla)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Spojení do jednoho balíčku
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# ==========================================
# 4. FINÁLNÍ PIPELINE (Předzpracování + Model)
# ==========================================
# ZDE MODELER ZKOUŠÍ RŮZNÉ ALGORITMY
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

# ==========================================
# 5. TRÉNOVÁNÍ A PREDIKCE
# ==========================================
# Vše se učí POUZE z trénovacích dat (žádný Data Leakage!)
model_pipeline.fit(X_train, y_train)

# Aplikace naučených pravidel a predikce na testovacích datech
y_pred = model_pipeline.predict(X_test)

print("Pipeline úspěšně proběhla! Model je natrénován.")